<a href="https://colab.research.google.com/github/wbc-mjlab/wbc-mjlab/blob/main/notebooks/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# wbc-mjlab demo (G1 whole-body tracking)

Interactive **Viser** play with the bundled checkpoint and **samples** clip library (13 motions).

Steps: clone → install → convert CSVs to NPZ → run `wbc-mjlab-demo`.

> **Runtime:** GPU (T4+). First run ~10–15 min (dependency sync + FK conversion).

## 1. GPU + headless MuJoCo

In [ ]:
import os

os.environ["MUJOCO_GL"] = "egl"

!nvidia-smi

## 2. Clone and install

In [ ]:
REPO_ROOT = "/content/wbc-mjlab"

if not os.path.isdir(REPO_ROOT):
  !git clone -q https://github.com/wbc-mjlab/wbc-mjlab.git {REPO_ROOT}

%cd {REPO_ROOT}

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = os.environ["HOME"] + "/.local/bin:" + os.environ["PATH"]

!uv sync --extra cu128
print("✓ wbc-mjlab ready")

## 3. Convert bundled samples (one time)

Source CSVs are in the repo; NPZ clips are generated locally under `data/g1/samples/npz/`.

In [ ]:
!uv run wbc-mjlab-data-to-npz --robot g1 --dataset samples --batch-size 16

## 4. Launch demo server

Runs bundled `demos/wbc_g1/model.pt` with 8 parallel envs and uniform clip sampling. Wait for the Viser URL message, then open the next cell.

In [ ]:
import subprocess
import sys

VISER_PORT = 8081

process = subprocess.Popen(
  ["uv", "run", "wbc-mjlab-demo"],
  cwd=REPO_ROOT,
  stdout=subprocess.PIPE,
  stderr=subprocess.STDOUT,
  universal_newlines=True,
  bufsize=1,
)

for line in process.stdout:
  print(line, end="")
  sys.stdout.flush()
  lower = line.lower()
  if "serving" in lower or "running on" in lower or str(VISER_PORT) in line:
    print("\n" + "=" * 50)
    print("✅ Viser server is up — run the next cell for the viewer iframe.")
    print("=" * 50)
    break

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(VISER_PORT)

## Manual play (optional)

```bash
uv run wbc-mjlab-play --task Wbc-G1 --dataset samples \
  --checkpoint-file demos/wbc_g1/model.pt --viewer viser --num-envs 8
```